
# Silver — URA

Atualiza a Silver de URA consumindo o **Change Data Feed** da Bronze como **fonte de
streaming** (`Trigger.AvailableNow`) e aplicando **MERGE** idempotente via
`foreachBatch`. O **checkpoint** do stream controla o progresso (offset/versão), com
garantia de *exactly-once*.

**Fluxo**
1. Lê o CDF da Bronze como stream, a partir do checkpoint.
2. Filtra mudanças (`insert`) e faz o parse do `body` (JSON) com `StructType`.
3. Renomeia colunas e deriva campos de data.
4. Garante a Silver com o mesmo schema e aplica **MERGE** por chave de negócio.



## Parâmetros


In [ ]:
# ===================== PARÂMETROS (Widgets) =====================
import sys

# Caminho da lib compartilhada (ajuste ao seu Repo/Workspace ou empacote como wheel).
sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("silver_schema", "s_dm_callcenter")
dbutils.widgets.text("bronze_table", "ura_once")
dbutils.widgets.text("silver_table", "tabe_ura_anlt")
dbutils.widgets.text("checkpoint_base", "/Volumes/prd/s_dm_callcenter/checkpoints/silver")

CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
BRONZE_TABLE  = dbutils.widgets.get("bronze_table")
SILVER_TABLE  = dbutils.widgets.get("silver_table")

BRONZE_FQN = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_FQN = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
CHECKPOINT = f"{dbutils.widgets.get('checkpoint_base').rstrip('/')}/{SILVER_TABLE}"

from pyspark.sql import functions as F, types as T
from transforms import SilverStream

print("Bronze:", BRONZE_FQN)
print("Silver:", SILVER_FQN)
print("Checkpoint:", CHECKPOINT)


## Transformação


In [ ]:

def transform(df_raw):
    # Schema do JSON
    schema = T.StructType([
        T.StructField("id_chamada", T.StringType(),  True),
        T.StructField("id_cliente", T.StringType(),  True),
        T.StructField("id_fila", T.StringType(),     True),
        T.StructField("data_hora_inicio", T.TimestampType(), True),
        T.StructField("data_hora_fim",    T.TimestampType(), True),
        T.StructField("autenticado", T.BooleanType(), True),
        T.StructField("opcoes_navegadas", T.IntegerType(), True),
        T.StructField("codigo_opcao", T.StringType(),  True),
        T.StructField("derivado_atendimento", T.BooleanType(), True),
    ])

    rename = {
        "id_chamada":          "ID_CHAM",
        "id_cliente":          "ID_CLIE",
        "id_fila":             "ID_FILA",
        "data_hora_inicio":    "DH_INIC",
        "data_hora_fim":       "DH_FIM",
        "opcoes_navegadas":    "QT_OPCA_NAVG",
        "codigo_opcao":        "CD_ULTI_OPCA",
        "autenticado":         "IN_AUTN",
        "derivado_atendimento":"IN_DERV_ATEN",
    }

    df = (df_raw
          .withColumn("body", F.from_json(F.col("body"), schema))
          .filter(F.col("body").isNotNull())
          .withColumn("_cv", F.col("_commit_version").cast("long"))
          .withColumn("_ct", F.col("_commit_timestamp").cast("timestamp"))
          .select("body.*", "_cv", "_ct")
    )

    # Renomeia as colunas
    df = df.select([F.col(c).alias(rename.get(c, c)) for c in df.columns])

    # Colunas de DATA do evento e carga
    df = (df
          .withColumn("CD_PERI", F.date_format(F.col("DH_INIC"), "yyyyMM").cast("int"))
          .withColumn("DT_INIC", F.to_date("DH_INIC"))
          .withColumn("DT_FIM",  F.to_date("DH_FIM"))
          .withColumn("DH_REFE_CRGA", F.current_timestamp())
    )
    return df



## ▶️ Execução
Lê o CDF da Bronze como stream (`AvailableNow`), filtra `insert`, transforma e aplica
`MERGE` idempotente via `foreachBatch`. O **checkpoint** controla o progresso.


In [ ]:
# Upsert incremental via streaming (AvailableNow) + foreachBatch(MERGE).
# O checkpoint controla o progresso (exactly-once).
stream = SilverStream(spark)
stream.run(
    source_fqn=BRONZE_FQN,
    target_fqn=SILVER_FQN,
    transform=transform,
    keys=["ID_CHAM"],
    checkpoint_location=CHECKPOINT,
    cluster_by=["CD_PERI", "DT_INIC", "ID_CHAM"],
)
print(f"[OK] Silver atualizada → {SILVER_FQN}")